In [6]:
from datetime import datetime, timedelta
from dotenv import load_dotenv
import os
import pandas as pd
import requests

In [7]:
class AlphavantageScrapper:
    def __init__(self) -> None:
        load_dotenv()
        self.api_key = os.getenv('ALPHAVANTAGE_KEY')
        
    def process_data_as_df(self,data, ticker:str):
        new_data = []
        for info in data:
            ticker_info = [item for item in info['ticker_sentiment'] if item['ticker'] == ticker][0]
            filtered_info = {
                'time': info['time_published'],
                'title': info['title'],
                'url': info['url'],
                'summary': info['summary'],
                'source': info['source_domain'],
                'overall_sentiment_score': float(info['overall_sentiment_score']),
                'overall_sentiment_label': info['overall_sentiment_label'],
                'ticker': ticker,
                'ticker_relevance_score': float(ticker_info['relevance_score']),
                'ticker_sentiment_score': float(ticker_info['ticker_sentiment_score']),
                'ticker_sentiment_label': ticker_info['ticker_sentiment_label']
            }
            new_data.append(filtered_info)
        df = pd.DataFrame(new_data)
        df.time = pd.to_datetime(df.time,format='%Y%m%dT%H%M%S')
        df.set_index('time', inplace=True)
        return df
    
    def fetch_news_data(self, ticker:str, open_time:datetime, close_time:datetime, limit:int=1000):
        open_time = open_time.strftime('%Y%m%dT%H%M')
        close_time = close_time.strftime('%Y%m%dT%H%M')
        url = f'https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers={ticker}&time_from={open_time}&time_to={close_time}&sort=EARLIEST&limit={limit}&apikey={self.api_key}'
        try:
            response = requests.get(url)
            if response.status_code == 200:
                data = response.json()
                if 'feed' in data:
                    return data['feed']
                else:
                    print(data)
                    raise Exception("API limit reached")
        except Exception as error:
            print(f"Error in fetching data: {error}")
            return None
        
        
    def create_news_df(self, ticker:str, open_time:datetime, close_time:datetime, limit:int=1000):
        news_df = pd.DataFrame()
        last_date = open_time
        previous_date = open_time
        while True:
            news_data = self.fetch_news_data(ticker, last_date, close_time, limit)
            if news_data is not None:
                news_data = self.process_data_as_df(news_data, ticker)
                news_df = pd.concat([news_df, news_data])
                last_date = news_data.index[-1]
                if previous_date == last_date:
                    break
                previous_date = last_date
            else:
                break
        news_df = news_df[~news_df.index.duplicated(keep='first')]
        return news_df
    

In [17]:

open_time = pd.to_datetime('2024-04-01')
close_time = pd.to_datetime('2024-09-01')

def create_ticker_news_df(ticker:str, open_time:datetime, close_time:datetime):
    aps = AlphavantageScrapper()
    news_df = aps.create_news_df(ticker, open_time, close_time)
    if len(news_df)>0:
        print(f"news data for {ticker} from {open_time} to {close_time} with length of {len(news_df)}")
        display(news_df.head())
        display(news_df.tail())
        news_df.to_csv(f'news/{ticker}_{open_time.strftime(format="%Y-%m-%d")}-{close_time.strftime(format="%Y-%m-%d")}.csv')
    return news_df

tickers = [
    #"BAC",   # Bank of America
    #"INTC",  # Intel
    #"XOM",   # Exxon Mobil
    #"PFE",   # Pfizer
    #"KO",    # Coca-Cola
    #"V",     # Visa
    #"DIS",   # Disney
    #"ORCL",  # Oracle
    #"F",     # Ford
    #"JNJ",   # Johnson & Johnson (Saúde)
    #"CVX",   # Chevron (Energia)
    #"WMT",   # Walmart (Consumo)
    #"VZ",    # Verizon (Telecomunicações)
    #"MCD",   # McDonald's (Consumo)
    #"GS",    # Goldman Sachs (Finanças)
    #"C",     # Citigroup (Finanças)
    #"BA",    # Boeing (Indústria)
    #"CAT",   # Caterpillar (Indústria)
    #"GE",    # General Electric (Indústria)
    #"DE",    # Deere & Company (Indústria)
]


for ticker in tickers:
    news_df = create_ticker_news_df(ticker, open_time, close_time)
    break

news data for DE from 2024-04-01 00:00:00 to 2024-09-01 00:00:00 with length of 195


,title,url,summary,source,overall_sentiment_score,overall_sentiment_label,ticker,ticker_relevance_score,ticker_sentiment_score,ticker_sentiment_label
time,,,,,,,,,,
2024-04-01 11:17:00,"The Zacks Analyst Blog Highlights Walmart, Lin...",https://www.zacks.com/stock/news/2248151/the-z...,"Walmart, Linde, IBM, Deere and Duke Energy are...",www.zacks.com,0.303960,Somewhat-Bullish,DE,0.143843,0.039440,Neutral
2024-04-01 21:50:20,Deere ( DE ) Suffers a Larger Drop Than the ...,https://www.zacks.com/stock/news/2248704/deere...,Deere (DE) reachead $404.14 at the closing of ...,www.zacks.com,0.155907,Somewhat-Bullish,DE,0.521961,0.151521,Somewhat-Bullish
2024-04-02 10:00:12,World's Billionaires List 2024: The Top 200,https://www.forbes.com/sites/chasewithorn/2024...,Here are the wealthiest people on Forbes' annu...,www.forbes.com,0.100811,Neutral,DE,0.009098,0.073322,Neutral
2024-04-03 07:00:00,Alamo Group Shows Rising Relative Strength; St...,https://www.investors.com/ibd-data-stories/ala...,Alamo Group Shows Rising Relative Strength. St...,www.investors.com,0.327126,Somewhat-Bullish,DE,0.358152,0.000000,Neutral
2024-04-03 19:45:24,Peering Into Deere's Recent Short Interest - D...,https://www.benzinga.com/insights/short-seller...,Deere's DE short percent of float has fallen 1...,www.benzinga.com,0.264538,Somewhat-Bullish,DE,0.212115,0.081887,Neutral


,title,url,summary,source,overall_sentiment_score,overall_sentiment_label,ticker,ticker_relevance_score,ticker_sentiment_score,ticker_sentiment_label
time,,,,,,,,,,
2024-08-27 11:30:00,Compact Wheel Loader Market is Projected to Ex...,https://www.benzinga.com/pressreleases/24/08/g...,"Rockville, MD, Aug. 27, 2024 ( GLOBE NEWSWIRE ...",www.benzinga.com,0.389879,Bullish,DE,0.034984,-0.054870,Neutral
2024-08-28 13:00:00,Precision Farming Market to Gain USD 22.09 Bil...,https://www.benzinga.com/pressreleases/24/08/g...,"Westford, USA, Aug. 28, 2024 ( GLOBE NEWSWIRE ...",www.benzinga.com,0.397551,Bullish,DE,0.050679,0.000000,Neutral
2024-08-28 16:30:58,Looking At Deere's Recent Unusual Options Acti...,https://www.benzinga.com/insights/options/24/0...,Investors with a lot of money to spend have ta...,www.benzinga.com,0.129174,Neutral,DE,0.853774,0.193453,Somewhat-Bullish
2024-08-30 12:02:17,Precision Farming Market to Gain USD 22.09 Bil...,https://www.benzinga.com/pressreleases/24/08/g...,"Westford, USA, Aug. 30, 2024 ( GLOBE NEWSWIRE ...",www.benzinga.com,0.397551,Bullish,DE,0.050679,0.000000,Neutral
2024-08-30 16:51:00,"Titan Machinery Meets on Q2 Earnings, Lowers F...",https://www.zacks.com/stock/news/2329928/titan...,TITN expects the EPS between breakeven and 50 ...,www.zacks.com,-0.111862,Neutral,DE,0.099476,-0.037952,Neutral


In [ ]:
pd.read_csv('news/MSFT_2024-04-01-2024-09-01.csv')

In [ ]:
df = pd.read_csv('news/oldGOOG_2024-04-01-2024-09-01.csv', index_col=0)
df.index = pd.to_datetime(df.index)
df

In [48]:
df2 = pd.concat([df,news_df])

df2 = df2[~df2.index.duplicated(keep='first')]

In [ ]:
df2

In [51]:
df2.to_csv("GOOG_2024-04-01-2024-09-01.csv", index=True)

In [ ]:
news_df

In [16]:
import requests
from dotenv import load_dotenv
import os

# Chave de API do AlphaVantage
api_key = os.getenv('ALPHAVANTAGE_KEY')

def test_news(ticker):
    # Ticker do ativo que deseja buscar notícias

    # URL da API da AlphaVantage para buscar notícias
    url = f'https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers={ticker}&apikey={api_key}'

    # Faz a requisição para a API
    response = requests.get(url)

    # Verifica se a requisição foi bem-sucedida
    if response.status_code == 200:
        data = response.json()
        if 'feed' in data:
            print("Sucesso na requisição para", ticker)
        else:
            print(f"Erro na requisição para {ticker}: ", data)


# Lista dos 20 ativos de maior volume no mercado americano
tickers = [
    "JNJ",   # Johnson & Johnson (Saúde)
    "CVX",   # Chevron (Energia)
    "WMT",   # Walmart (Consumo)
    "VZ",    # Verizon (Telecomunicações)
    "MCD",   # McDonald's (Consumo)
    "GS",    # Goldman Sachs (Finanças)
    "C",     # Citigroup (Finanças)
    "BA",    # Boeing (Indústria)
    "CAT",   # Caterpillar (Indústria)
    "GE",    # General Electric (Indústria)
    "DE",    # Deere & Company (Indústria)
]

for tck in tickers:
    test_news(tck)

Erro na requisição para JPM:  {'Information': 'Invalid inputs. Please refer to the API documentation https://www.alphavantage.co/documentation#newsapi and try again.'}
Sucesso na requisição para JNJ
Sucesso na requisição para CVX


KeyboardInterrupt: 

In [ ]:
tickers_non_tech = [
    "JNJ",   # Johnson & Johnson (Saúde)
    "CVX",   # Chevron (Energia)
    "WMT",   # Walmart (Consumo)
    "VZ",    # Verizon (Telecomunicações)
    "MCD",   # McDonald's (Consumo)
    "GS",    # Goldman Sachs (Finanças)
    "C",     # Citigroup (Finanças)
    "BA",    # Boeing (Indústria)
    "CAT",   # Caterpillar (Indústria)
    "GE",    # General Electric (Indústria)
    "DE",    # Deere & Company (Indústria)
]

# Print para verificar os tickers
print(tickers_non_tech)


In [ ]:
tickers = [
    "BAC",   # Bank of America
    "INTC",  # Intel
    "XOM",   # Exxon Mobil
    "PFE",   # Pfizer
    "KO",    # Coca-Cola
    "V",     # Visa
    "DIS",   # Disney
    "ORCL",  # Oracle
    "F",     # Ford
    "NKE"    # Nike,
    "JNJ",   # Johnson & Johnson (Saúde)
    "CVX",   # Chevron (Energia)
    "WMT",   # Walmart (Consumo)
    "VZ",    # Verizon (Telecomunicações)
    "MCD",   # McDonald's (Consumo)
    "GS",    # Goldman Sachs (Finanças)
    "C",     # Citigroup (Finanças)
    "BA",    # Boeing (Indústria)
    "CAT",   # Caterpillar (Indústria)
    "GE",    # General Electric (Indústria)
    "DE",    # Deere & Company (Indústria)
]